# Data Generation



## 1) SOBOL Samling

Imports for numerical, tabular, faker text, plotting, and optional COS-JAX generation.

In [2]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
from scipy.stats import qmc
from heston_project.pricing.COS_with_jax_gradients import *

#Jacob
try:
    import lets_be_rational.LetsBeRational as lbr
    print("Loaded via package import")

except ModuleNotFoundError:
    # Build the absolute path starting from your Mac's User folder (~)
    so_path = Path.home() / "Kandidatarbete/venv/lib/python3.13/site-packages/lets_be_rational/_LetsBeRational.cpython-313-darwin.so"

    if not so_path.exists():
        raise FileNotFoundError(f"Still can't find it! I am looking exactly here: {so_path}")

    # Load the C-extension directly
    spec = importlib.util.spec_from_file_location("_LetsBeRational", so_path)
    lbr = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(lbr)

    print("Loaded via direct .so import")
    print("Final module:", lbr)


'''
KÖR pip install -e . för att få tillgång till pricing!!!
'''


PARAM_ORDER = ["kappa", "theta", "sigma", "rho", "v0", "T", "r", "lm"]

#bounds for Sobol scaling, in PARAM_ORDER.
DEFAULT_LOWER = np.array([0.05, 0.0, 0.1, -0.95, 0.0, 1/52, -0.01, -1.5], dtype=float)
DEFAULT_UPPER = np.array([5.0, 1.0, 1.5, 0.00, 1.0, 3.0, 0.1, 1.5], dtype=float)
 
def next_power_two(n: int) -> int:
    if n <= 1:
        return 1
    return 1 << (n - 1).bit_length()

def sample_parameters_sobol(n_rows: int, seed_value: int) -> pd.DataFrame:
    #Sobol sampling för alla parametrar
    sobol = qmc.Sobol(d=len(PARAM_ORDER), scramble=True, seed=seed_value)
    m = int(np.log2(next_power_two(n_rows)))
    unit = sobol.random_base2(m=m)[:n_rows]
    scaled = qmc.scale(unit, DEFAULT_LOWER, DEFAULT_UPPER)
    df = pd.DataFrame(scaled, columns=PARAM_ORDER)
    df["S"] = 1.0
    df["K"] = np.exp(-df["lm"])
    return df

Loaded via package import


## 2) COS-JAX

price and price greeks

## Sobol Discrepancy Test

Test the uniformity and distribution quality of Sobol samples.


In [3]:
# Test Sobol discrepancy on sample_parameters_sobol output
from scipy.stats.qmc import discrepancy

# Sample parameters using the actual function
n_test = 20000
sobol_samples_df = sample_parameters_sobol(n_rows=n_test, seed_value=42)

# Extract only the sampled parameters (exclude S, K which are derived)
sobol_samples = sobol_samples_df[PARAM_ORDER].values

# Normalize to [0, 1] for discrepancy calculation
sobol_samples_normalized = np.zeros_like(sobol_samples)
for i, param in enumerate(PARAM_ORDER):
    sobol_samples_normalized[:, i] = (sobol_samples[:, i] - DEFAULT_LOWER[i]) / (DEFAULT_UPPER[i] - DEFAULT_LOWER[i])

# Calculate discrepancy
disc = discrepancy(sobol_samples_normalized)

print(f"Sobol Discrepancy Test ({n_test} samples from sample_parameters_sobol):")
print(f"  Discrepancy value: {disc:.6f}")
print(f"  (Lower is better - indicates more uniform distribution)")


Sobol Discrepancy Test (20000 samples from sample_parameters_sobol):
  Discrepancy value: 0.000003
  (Lower is better - indicates more uniform distribution)


In [3]:
### importat ist...

## 3) Computing price and grad (till dataframe)


In [4]:
#Pris diskonterat och absolut
#Pris-greeks då också diskonterade absoluta
def compute_price_and_grads(df: pd.DataFrame, N: int, L: int, q: float = 0.0):
    #Beräknar COS-JAX price och gradients och appendar columns till DF.
    out = df.copy()

    # S is fixed at 1.0, K is computed from lm
    out["S"] = 1.0
    out["K"] = np.exp(-out["lm"])

    n = len(out)
    prices = np.full(n, np.nan, dtype=float)
    dtheta = np.full(n, np.nan, dtype=float)
    dv0 = np.full(n, np.nan, dtype=float)
    dsigma = np.full(n, np.nan, dtype=float)
    drho = np.full(n, np.nan, dtype=float)
    dkappa = np.full(n, np.nan, dtype=float)
    stable = np.zeros(n, dtype=bool)

    for i, row in out.iterrows():
        params = jnp.array(
            [
                float(row["theta"]),
                float(row["v0"]),
                float(row["sigma"]),
                float(row["rho"]),
                float(row["kappa"]),
            ],
            dtype=jnp.float64,
        )

        x = float(row["lm"])
        T = float(row["T"])
        r = float(row["r"])

        try:
            p = price_jit(params, x, T, r, float(q), int(N), int(L))
            g = grad_price(params, x, T, r, float(q), int(N), int(L))
            g_np = np.array(g, dtype=float)

            prices[i] = float(p)
            dtheta[i] = g_np[0]
            dv0[i] = g_np[1]
            dsigma[i] = g_np[2]
            drho[i] = g_np[3]
            dkappa[i] = g_np[4]
            stable[i] = np.isfinite(prices[i]) and np.all(np.isfinite(g_np))
        except Exception:
            stable[i] = False

    out["price"] = prices
    out["dtheta"] = dtheta
    out["dv0"] = dv0
    out["dsigma"] = dsigma
    out["drho"] = drho
    out["dkappa"] = dkappa
    out["stable"] = stable

    return out


## 4) Batchar prissättning (Allt i DataFrame)



In [5]:
# --- Config ---
n_samples = 200000       # Total antal rader att generera
batch_size = 200        # Prissätt 200 rader åt gången
N = 2048*2*2*2*2  # COS expansion N
L = 12                  # COS truncation L
q = 0.0                 # dividend yield
seed = 42

# 1) Sampla ALLA parametrar först (Sobol)
print(f"Sampling {n_samples} rows with Sobol...")
df_all_params = sample_parameters_sobol(n_rows=n_samples, seed_value=seed)
print(f"✓ Sampled {len(df_all_params)} rows")

# 2) Batch-process: prissätt 200 rader åt gången
stable_total = 0
all_results = []

for start in range(0, len(df_all_params), batch_size):
    end = min(start + batch_size, len(df_all_params))
    chunk = df_all_params.iloc[start:end].copy().reset_index(drop=True)
    
    # Preallocate arrays för batch
    prices = np.full(len(chunk), np.nan, dtype=float)
    grads = np.full((len(chunk), 5), np.nan, dtype=float)
    stable = np.zeros(len(chunk), dtype=bool)
    
    for i, row in chunk.iterrows():
        params = jnp.array([
            float(row["theta"]),
            float(row["v0"]),
            float(row["sigma"]),
            float(row["rho"]),
            float(row["kappa"]),
        ], dtype=jnp.float64)
        
        try:
            p = price_jit(params, float(row["lm"]), float(row["T"]), float(row["r"]), float(q), int(N), int(L))
            g = grad_price(params, float(row["lm"]), float(row["T"]), float(row["r"]), float(q), int(N), int(L))
            g_np = np.array(g, dtype=float)
            
            prices[i] = float(p)
            grads[i, :] = g_np
            stable[i] = np.isfinite(prices[i]) and np.all(np.isfinite(g_np))
        except Exception:
            stable[i] = False
    
    # Appenda pris och greeks till chunk DataFrame
    chunk["price"] = prices
    chunk["dtheta"] = grads[:, 0]
    chunk["dv0"] = grads[:, 1]
    chunk["dsigma"] = grads[:, 2]
    chunk["drho"] = grads[:, 3]
    chunk["dkappa"] = grads[:, 4]
    chunk["stable"] = stable
    
    stable_total += int(stable.sum())
    all_results.append(chunk)
    
    # Progress output
    pct = 100.0 * end / len(df_all_params)
    print(f"  Batch {start}:{end} ({pct:.1f}% klar, stable: {int(stable.sum())}/{len(chunk)})")

# 3) Kombinera alla batchar till en enda DataFrame
df_final = pd.concat(all_results, ignore_index=True)

#### TA BORT SEN!!!
neg = df_final["price"] < 0
print("Antal negativa priser:", neg.sum())
print("Minsta pris:", df_final.loc[neg, "price"].min())
###

print("\n✓ Done!")
print(f"DataFrame shape: {df_final.shape}")
print(df_final.head())


Sampling 200000 rows with Sobol...
✓ Sampled 200000 rows
  Batch 0:200 (0.1% klar, stable: 200/200)
  Batch 200:400 (0.2% klar, stable: 200/200)
  Batch 400:600 (0.3% klar, stable: 200/200)
  Batch 600:800 (0.4% klar, stable: 200/200)
  Batch 800:1000 (0.5% klar, stable: 200/200)
  Batch 1000:1200 (0.6% klar, stable: 200/200)
  Batch 1200:1400 (0.7% klar, stable: 200/200)
  Batch 1400:1600 (0.8% klar, stable: 200/200)
  Batch 1600:1800 (0.9% klar, stable: 200/200)
  Batch 1800:2000 (1.0% klar, stable: 200/200)
  Batch 2000:2200 (1.1% klar, stable: 200/200)
  Batch 2200:2400 (1.2% klar, stable: 200/200)
  Batch 2400:2600 (1.3% klar, stable: 200/200)
  Batch 2600:2800 (1.4% klar, stable: 200/200)
  Batch 2800:3000 (1.5% klar, stable: 200/200)
  Batch 3000:3200 (1.6% klar, stable: 200/200)
  Batch 3200:3400 (1.7% klar, stable: 200/200)
  Batch 3400:3600 (1.8% klar, stable: 200/200)
  Batch 3600:3800 (1.9% klar, stable: 200/200)
  Batch 3800:4000 (2.0% klar, stable: 200/200)
  Batch 4000:4

In [6]:
# spara ner till CSV fil
from pathlib import Path

out_dir = Path("../../data")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"dataset_pricing_and_greeks_{n_samples}(1).csv"
df_final.to_csv(out_path, index=False)
print(f"✓ Sparade {len(df_final)} rader till {out_path.resolve()}")

✓ Sparade 200000 rader till C:\Users\linne\Documents\GitHub\Kandidatarbete\data\dataset_pricing_and_greeks_200000(1).csv


## 4.1) Tar bort rader med negativa priser

In [54]:

df_final = pd.read_csv('../../data/dataset_pricing_and_greeks_200000(1).csv')

# Identifiera negativa priser för loggning
neg_mask = df_final["price"] < 0
print(f"Antal rader med negativa priser som tas bort: {neg_mask.sum()} / {len(df_final)}")

# Behåll endast rader där priset är 0.0 eller högre
df_final = df_final[df_final["price"] >= 0].copy()

# Återställ index så att vi inte har luckor från de borttagna raderna
df_final.reset_index(drop=True, inplace=True)

print(f"Antal rader kvar: {len(df_final)}")
print(f"Minsta pris nu: {df_final['price'].min()}")

Antal rader med negativa priser som tas bort: 171 / 200000
Antal rader kvar: 199829
Minsta pris nu: 0.0


## 4.2) Sätt arbitrage priser till NaN

In [55]:
# Parametrar
S = df_final["S"].to_numpy()
K = df_final["K"].to_numpy()
T = df_final["T"].to_numpy()
r = df_final["r"].to_numpy()
prices = df_final["price"].to_numpy()

# Diskonterade arbitragegränser för put (q = 0)
lower_disc = np.maximum(K * np.exp(-r * T) - S, 0.0)
upper_disc = K * np.exp(-r * T)

eps = 1e-12

# Alla brott mot bounds
any_lower_violation = prices < lower_disc
any_upper_violation = prices > upper_disc

# Små brott (inom eps)
tiny_lower_violation = (prices < lower_disc) & (prices >= lower_disc - eps)
tiny_upper_violation = (prices > upper_disc) & (prices <= upper_disc + eps)

# Riktiga brott (mer än eps)
real_lower_violation = prices < (lower_disc - eps)
real_upper_violation = prices > (upper_disc + eps)

print(f"Uppfyller lower bound: {(prices >= lower_disc).sum()} / {len(prices)}")
print(f"Uppfyller upper bound: {(prices <= upper_disc).sum()} / {len(prices)}")

print(f"Alla lower-brott: {any_lower_violation.sum()} / {len(prices)}")
print(f"Små lower-brott inom eps: {tiny_lower_violation.sum()} / {len(prices)}")
print(f"Riktiga lower-brott > eps: {real_lower_violation.sum()} / {len(prices)}")

print(f"Alla upper-brott: {any_upper_violation.sum()} / {len(prices)}")
print(f"Små upper-brott inom eps: {tiny_upper_violation.sum()} / {len(prices)}")
print(f"Riktiga upper-brott > eps: {real_upper_violation.sum()} / {len(prices)}")

'''# Sätt riktiga brott till NaN
real_violation_mask = real_lower_violation | real_upper_violation
df_final.loc[real_violation_mask, "price"] = np.nan'''

# Radera rader med riktiga brott
real_violation_mask = real_lower_violation | real_upper_violation
deleted_count = real_violation_mask.sum()
df_final = df_final[~real_violation_mask].copy()

# Återställ index så att vi inte har luckor från de borttagna raderna
df_final.reset_index(drop=True, inplace=True)

print(f"Antal riktiga brott raderade: {deleted_count}")
print(f"Antal rader kvar: {len(df_final)}")


print(f"Antal riktiga brott satta till raderade: {real_violation_mask.sum()}")


Uppfyller lower bound: 199278 / 199829
Uppfyller upper bound: 199829 / 199829
Alla lower-brott: 551 / 199829
Små lower-brott inom eps: 542 / 199829
Riktiga lower-brott > eps: 9 / 199829
Alla upper-brott: 0 / 199829
Små upper-brott inom eps: 0 / 199829
Riktiga upper-brott > eps: 0 / 199829
9
Antal riktiga brott raderade: 9
Antal rader kvar: 199820
Antal riktiga brott satta till raderade: 9


In [57]:
from pathlib import Path

out_dir = Path("../../data")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"dataset_arbitrage_fix_200000(2).csv"
df_final.to_csv(out_path, index=False)

print(f"Sparade {len(df_final)} rader till {out_path.resolve()}")
print(f"Kolumner: {list(df_final.columns)}")
print(np.sum(np.isnan(df_final["price"])))
print(df_final[df_final["price"].isna()])

Sparade 199820 rader till /Users/jacobhansen/Desktop/Kanarb/Kandidatarbete/data/dataset_arbitrage_fix_200000(2).csv
Kolumner: ['kappa', 'theta', 'sigma', 'rho', 'v0', 'T', 'r', 'lm', 'S', 'K', 'price', 'dtheta', 'dv0', 'dsigma', 'drho', 'dkappa', 'stable']
0
Empty DataFrame
Columns: [kappa, theta, sigma, rho, v0, T, r, lm, S, K, price, dtheta, dv0, dsigma, drho, dkappa, stable]
Index: []


In [67]:
#spara ner till CSV fil
from pathlib import Path

out_dir = Path("../../data")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"dataset_arbitrage_fix_200000(2).csv"
df_final.to_csv(out_path, index=False)

# 5) IV med Let's Be Rational

In [ ]:
df_final = pd.read_csv('../../data/dataset_arbitrage_fix_200000(2).csv')
n_samples = len(df_final)

lm  = df_final["lm"].to_numpy()                 # lm = ln(S/K)
T   = df_final["T"].to_numpy()
r   = df_final["r"].to_numpy()
S   = df_final["S"].to_numpy() 
K   = df_final["K"].to_numpy()
price = df_final["price"].to_numpy()            # diskonterat och absolut
P_forward  = price * np.exp(r * T)              # odiskonterat och absolut put-pris då lets be rational förväntar sig den input formen        
q = 0.0                                         # utdelning
qflag = -1                                      # Put-flagga -1 i let´s be rational bibloteket

m = lm + (r - q) * T                            # m = ln(F/K)
F = np.exp(m) * K 

lower = np.maximum(K - F, 0.0)                  # Arbitragegränser. Görs i forward form då det är inputen i lbr
upper = K
eps = 1e-12

#iv_low_mask = (P_forward - lower) <= eps                        # Price nära intrisic -> IV = 0, för numreriskt brus vilket gör lbr instabil och kan generera galna iv
#iv[iv_low_mask] = 0.0                                                 # Nu är masken dock för "bred" och fångar även rimliga iv. Att minska den hjälper ej då de galna ej lågger linjärt längre in       
#iv[iv_unreal_mask] = np.nan
#print(f"IV's -> 0 : {len(iv[iv_low_mask])} / {n_samples}")
                                                                       
iv_unreal_mask = (P_forward < lower) | (P_forward > upper)             # Inget reelt IV när pris är utanför detta

#iv_low_mask = (P_forward - lower) <= eps                                               # P_undisc nära intrisic -> IV = 0, för numreriskt brus vilket gör lbr instabil och kan generera galna iv
                                                                                        # Nu är masken dock för "bred" och fångar även rimliga iv. Att minska den hjälper ej då de galna ej lågger linjärt längre in
iv_unreal_mask = ~np.isfinite(P_forward) | (P_forward < lower) | (P_forward > upper)    # Inget reelt IV när pris är utanför detta. Sätter deras IV till NaN
iv = np.full(len(P_forward), np.nan, dtype=float)

print(f"Arbitrage IV's (satta till NaN): {len(iv[iv_unreal_mask])} / {n_samples}")
print(f"Arbitrage IV's under intrisic: {(P_forward < lower).sum()} / {n_samples}")

# Kör Let’s Be Rational
for i, (P_i, F_i, K_i, T_i) in enumerate(zip(P_forward, F, K, T)):
    if iv_unreal_mask[i]:
        continue
    iv[i] = lbr.implied_volatility_from_a_transformed_rational_guess(P_i, F_i, K_i, T_i, qflag)

df_final["IV"] = iv                             # IV diskonteras/normaliseras ej utan är endast volatilitetsskala
print("Min IV:", np.nanmin(iv))
print("Max IV:", np.nanmax(iv))
print("Antal NaN IV:", np.sum(np.isnan(iv)))


#Tester för epsilon & masker:
print(" - Vad händer i en mask runt intrisic:") #lägg till eps/masken för att se vilka ivs som beräknas
gap = P_forward - lower

near_intrinsic = np.abs(gap) <= eps 

print("Antal priser nära intrisic:", near_intrinsic.sum())
iv_near = iv[near_intrinsic]

print("Antal NaN IV nära intrisic:", np.sum(np.isnan(iv_near)))
print("Min IV nära intrisic:", np.nanmin(iv_near))
print("Max IV nära intrisic:", np.nanmax(iv_near))
print("Percentiler IV:", np.nanpercentile(iv_near, [0, 1, 5, 50, 95]))

low_idx = np.where(near_intrinsic)[0]

print("Antal under lower inom low-mask:", np.sum(P_forward[low_idx] < lower[low_idx]))
print("Antal över/på lower inom low-mask:", np.sum(P_forward[low_idx] >= lower[low_idx]))

print("Gap i low-mask, min/max:")
print(np.min(gap[low_idx]), np.max(gap[low_idx]))

valid = np.isfinite(iv_near) & (iv_near > 0)

print("Median (valid IV):", np.median(iv_near[valid]))
print("95% (valid IV):", np.percentile(iv_near[valid], 95))

Arbitrage IV's (satta till NaN): 539 / 199820
Arbitrage IV's under intrisic: 539 / 199820
Min IV: 0.0
Max IV: 1.4097583039804613
 - Vad händer i en mask runt intrisic:
Antal priser nära intrisic: 2237
Antal NaN IV nära intrisic: 539
Min IV nära intrisic: 0.0
Max IV nära intrisic: 1.4097583039804613
Percentiler IV: [0.         0.         0.         0.51343263 1.05349732]
Antal under lower inom low-mask: 539
Antal över/på lower inom low-mask: 1698
Gap i low-mask, min/max:
-4.973799150320701e-14 9.952039192739903e-13
Median (valid IV): 0.5410793526081219
95% (valid IV): 1.0624898111710779


# 6) Vega

In [69]:
# Vega = F * phi(d1) * sqrt(T)

mask = (np.isfinite(iv) & (iv > 0) &         # Markerar rader där IV, F, T är ändliga och positiva (boolean mask)
        np.isfinite(F) & (F > 0.0) &             # Endast iv mask som gör skillnad för vårt dataset, 121 st iv=0
        np.isfinite(T) & (T > 0) &
        np.isfinite(K) & (K > 0))                  

d1 = np.full_like(iv, np.nan)                                                                      # d1[mask] = (np.log(F[mask]) + 0.5 * iv[mask]**2 * T[mask]) / (iv[mask] * np.sqrt(T[mask]))
d1[mask] = (np.log(F[mask]/K[mask]) + 0.5 * iv[mask]**2 * T[mask]) / (iv[mask] * np.sqrt(T[mask])) # Antar raden över K=1? Isf är detta riktiga formeln?

inv_sqrt_2pi = 1.0 / np.sqrt(2*np.pi)
phi_d1 = np.full_like(iv, np.nan)
phi_d1[mask] = inv_sqrt_2pi * np.exp(-0.5 * d1[mask]**2)

vega = np.full_like(iv, np.nan)
vega[mask] = np.exp(-r[mask] * T[mask]) * F[mask] * phi_d1[mask] * np.sqrt(T[mask])   # Diskonterat (som pris greeksen)

df_final["vega"] = vega


# Filter för att undvika extrema IV-greeks
#Kanske vi inte borde göra?
'''iv_floor = 1e-12
vega_floor = 1e-4

good = (
    np.isfinite(df_final["IV"].to_numpy()) & (df_final["IV"].to_numpy() > iv_floor) &
    np.isfinite(df_final["vega"].to_numpy()) & (df_final["vega"].to_numpy() > vega_floor)
)'''

#Här är utan filter

# good = (
#     np.isfinite(df_final["IV"].to_numpy()) & 
#     np.isfinite(df_final["vega"].to_numpy())
# )


# print("Filtered out:", (~good).sum(), "of", len(good))

# df_final = df_final.loc[good].reset_index(drop=True)

'iv_floor = 1e-12\nvega_floor = 1e-4\n\ngood = (\n    np.isfinite(df_final["IV"].to_numpy()) & (df_final["IV"].to_numpy() > iv_floor) &\n    np.isfinite(df_final["vega"].to_numpy()) & (df_final["vega"].to_numpy() > vega_floor)\n)'

# 7) IV-Greeks

In [70]:
for k in PARAM_ORDER:
    if f"d{k}" not in df_final.columns:
        continue

    df_final[f'IVd{k}'] = df_final[f'd{k}'] / df_final['vega']

pd.set_option('display.max_columns', None)
print(df_final.head())
print(len(df_final))


      kappa     theta     sigma       rho        v0         T         r  \
0  2.183596  0.814368  1.228978 -0.899216  0.297939  2.522713  0.066755   
1  4.713177  0.127980  0.744998 -0.004147  0.941122  1.344923 -0.007963   
2  3.044741  0.547033  1.146860 -0.535639  0.612305  0.383218  0.093633   
3  0.158485  0.487110  0.126945 -0.367753  0.126601  2.180375  0.020593   
4  0.820252  0.742904  0.480596 -0.325174  0.109324  2.956465  0.074169   

         lm    S         K     price    dtheta       dv0    dsigma      drho  \
0  0.311887  1.0  0.732064  0.215761  0.162133  0.034450 -0.018084  0.014579   
1 -0.452794  1.0  1.572701  0.677077  0.337691  0.062981 -0.002005  0.026728   
2 -0.877387  1.0  2.404607  1.323848  0.011446  0.015000 -0.002666  0.013269   
3  1.393110  1.0  0.248302  0.001889  0.004390  0.025505  0.007158 -0.001558   
4 -1.309854  1.0  3.705632  2.135614  0.278432  0.158386 -0.065740  0.089448   

     dkappa  stable        IV      vega  IVdkappa  IVdtheta  IVdsigm

In [71]:
for k in PARAM_ORDER:
    if f"d{k}" not in df_final.columns:
        continue

    df_final[f'IV d{k}'] = df_final[f'd{k}'] / df_final['vega']

pd.set_option('display.max_columns', None)
print(df_final.head())
print(len(df_final))


# 8) Spara till CSV

In [72]:
from pathlib import Path

out_dir = Path("../../data")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"full_dataset_training_200000(2).csv"
df_final.to_csv(out_path, index=False)

print(f"Sparade {len(df_final)} rader till {out_path.resolve()}")
print(f"Kolumner: {list(df_final.columns)}")

Sparade 199820 rader till C:\Users\linne\Documents\GitHub\Kandidatarbete\data\full_dataset_training_200000(2).csv
Kolumner: ['kappa', 'theta', 'sigma', 'rho', 'v0', 'T', 'r', 'lm', 'S', 'K', 'price', 'dtheta', 'dv0', 'dsigma', 'drho', 'dkappa', 'stable', 'IV', 'vega', 'IVdkappa', 'IVdtheta', 'IVdsigma', 'IVdrho', 'IVdv0']
